In [ ]:
import sys
import pandas as pd
import pickle
import importlib

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../../')
sys.path.insert(0, '../../../../../')

# Add path to the other project src to handle imports with dashes in folder names
sys.path.insert(0, '../../../../../probabilistic_suffix_prediction_U-ED-LSTM/src')

In [ ]:
!ls ../../../../data/data/helpdesk.csv

In [ ]:
# log as csv
event_log_path = '../../../../data/data/helpdesk.csv'
event_log_df = pd.read_csv(event_log_path)

In [ ]:
!ls ../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/petri_net/helpdesk.pkl

In [ ]:
# import petri net, 
petri_net_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/petri_net/helpdesk.pkl'
with open(petri_net_path, 'rb') as f:
    net, im, fm = pickle.load(f)

# split csv data
train_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_train.csv'
helpdesk_train_df = pd.read_csv(train_csv_path)

val_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_val.csv'
helpdesk_val_df = pd.read_csv(val_csv_path)

test_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_test.csv'
helpdesk_test_df = pd.read_csv(test_csv_path)

# get case ids
unique_list_train = helpdesk_train_df["CaseID"].dropna().unique().tolist()
unique_list_val = helpdesk_val_df["CaseID"].dropna().unique().tolist()

case_ids = list(dict.fromkeys(unique_list_train + unique_list_val))

In [ ]:
import decision_mining.decision_discovery
importlib.reload(decision_mining.decision_discovery)
from decision_mining.decision_discovery import DecisionDiscovery

dd = DecisionDiscovery(petri_net=(net, im, fm),
                       event_log_df=event_log_df,
                       case_ids=case_ids,
                       case_id_key='CaseID',
                       activity_key='Activity',
                       time_key='CompleteTimestamp')

dynamic_attributes = ['Resource']
static_attributes = ['VariantIndex', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup']

decision_results = dd.decision_mining_datasets(attributes=dynamic_attributes,
                                               trace_attributes=static_attributes)

In [ ]:
xor_points = decision_results.decision_points.get('xor')
xor_tress = decision_results.decision_trees.get('xor')

print(xor_points)
print(list(xor_tress.values())[0])

dd.visualize_decision_tree(decision_tree=list(xor_tress.values())[0])